In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
display(df.describe()) # for numerical features
print("----------------------------------------------------------------------------------------------")
display(df.describe(include = "object")) # for categorical features


In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])
display(df.head())

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)
print("Removing all rows where target (Delivery_Time) is null/missing")
df = df.dropna(subset=['Delivery_Time']) # removing all rows where target (Delivery_Time) is null/missing
check_missing_values(df)
null_columns = df.columns[df.isna().any()] # removing all other rows where there is a missing value for a feature
df = df.dropna(subset=null_columns)
check_missing_values(df)

In [ ]:
# THIS IS IF USER PREFERS TO REPLACE MISSING VALUES FOR MEAN / MODE (mean for numerical columns & mode for categorical columns)
# null_columns = df.columns[df.isna().any()]
# print(null_columns)
# print('Columns with NaN values are:', null_columns)

# #fill NaN values with mean or mode (mean for numerical columns & mode for categorical columns)
# for c in null_columns:
#   value = df[c].mean() if df[c].dtype!='object' else df[c].mode()
#   df[c].fillna(value,inplace=True)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))
# Apply one-hot encoding to these categorical features using pd.get_dummies
df = pd.get_dummies(df, columns=categorical_cols, drop_first=False)
df = df.astype(float)
df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import MinMaxScaler
features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET
scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:
print("This cell is not needed as target imbalance exists in classification problems and not regression")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200)
}
all_results = {}
predictions = []
lr_mae = []
for name in models:
  all_results[name] = {'MAE': []}

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    predictions.append(y_pred)
    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)

    # Store results
    all_results[model_name]["MAE"].append(mae)
    lr_mae.append(mae)

print("Linear Regression Results")
print(f"Average MAE across all folds: {np.mean(lr_mae):.4f}")

In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs["Random Forest Regressor"] = models["Random Forest Regressor"].feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
predictions_ndarry = np.concatenate((predictions[0], predictions[1], predictions[2], predictions[3], predictions[4]))
%matplotlib inline
import matplotlib.pyplot as plt

plt.figure(figsize=(5,5))
plt.hist(predictions_ndarry, bins=30, edgecolor='black')
plt.show()

In [ ]:
# Task Bonus: Write your code here:
